In [ ]:
import time
notebook_start_time = time.time()

NOTICE: Code is optimized for the A100 GPU

# 1. Installing, importing libraries and making configurations

In [ ]:
!pip install uv

In [ ]:
!uv pip install torch numpy matplotlib lightning huggingface_hub triton pandas optuna optuna-integration[pytorch_lightning]

In [ ]:
!uv pip install mamba-ssm --no-build-isolation-package mamba-ssm

In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


import lightning as L
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download, login
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

from mamba_ssm import Mamba

import optuna
from optuna.integration import PyTorchLightningPruningCallback

In [ ]:
NUM_WORKERS = 4

NUM_EPOCHS = 10

# 2. Models developed

In [ ]:
def label_smoothed_bce_loss(logits, labels, eps=0.1):
    ce_hard = F.cross_entropy(logits, labels)
    log_probs = F.log_softmax(logits, dim=-1)
    ce_soft = -log_probs.mean(dim=-1).mean()
    return (1.0 - eps) * ce_hard + eps * ce_soft

In [ ]:
def get_optimizer(model, lr=1e-3, base_wd=0.01, dt_proj_wd=1e-3):
    dt_proj_weights = []
    other_params    = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'dt_proj.weight' in name:
            dt_proj_weights.append(param)
        else:
            other_params.append(param)
    param_groups = [
        {'params': other_params,    'weight_decay': base_wd,    'name': 'default'},
        {'params': dt_proj_weights, 'weight_decay': dt_proj_wd, 'name': 'dt_proj'},
    ]
    return torch.optim.AdamW(param_groups, lr=lr)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, d_model, N, expand=2, **kwargs):
        super().__init__()
        self.norm  = RMSNorm(d_model)
        self.mamba = Mamba(d_model=d_model, d_state=N, expand=expand)

    def forward(self, x):
        return x + self.mamba(self.norm(x))

In [ ]:
class MambaClassifier(L.LightningModule):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_layers: int = 2,
        N: int = 16,
        num_classes: int = 2,
        lr: float = 1e-3,
        label_smoothing: float = 0.0,
        dt_proj_wd: float = 0.0,
        expand: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.label_smoothing = label_smoothing
        self.dt_proj_wd = dt_proj_wd
        self.num_classes = num_classes

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            ResidualBlock(d_model, N, expand=expand) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

        # Compile Mamba layers for fused Triton kernels (replaces causal-conv1d)
        for i, layer in enumerate(self.layers):
            self.layers[i] = torch.compile(layer)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)               # (B, L, D)

        for layer in self.layers:
            x = layer(x)
        x = self.norm_f(x)

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            x = x.mean(dim=1)

        return self.classifier(x)

    def _shared_step(self, batch, stage):
        logits = self(batch['input_ids'], batch['attention_mask'])
        labels = batch['labels']

        if self.label_smoothing > 0:
            loss = label_smoothed_bce_loss(logits, labels, eps=self.label_smoothing)
        else:
            loss = F.cross_entropy(logits, labels)

        preds = logits.argmax(dim=-1)
        acc = (preds == labels).float().mean()

        self.log(f'{stage}_loss', loss, prog_bar=True)
        self.log(f'{stage}_acc',  acc,  prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, 'val')

    def test_step(self, batch, batch_idx):
        return self._shared_step(batch, 'test')

    def configure_optimizers(self):
        if self.dt_proj_wd > 0:
            return get_optimizer(
                self, lr=self.lr, base_wd=0.01, dt_proj_wd=self.dt_proj_wd
            )
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=0.01)

# 3. Data and running each model in its benchmark

## 3.0 Data and others

In [ ]:
repo_id = "monteirot/lra-benchmarks"

files = ['listops.pt', 'imdb_lra.pt', 'acl_retrieval_lra.pt', 'cifar10_sequential_lra.pt']

for f in files:
    print(f"Downloading {f}...")
    hf_hub_download(repo_id=repo_id, filename=f, repo_type="dataset", local_dir=".")

In [ ]:
def make_trainer():
    return L.Trainer(
        max_epochs=NUM_EPOCHS,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        logger=CSVLogger(save_dir="logs", name="lra-benchmarks"),
    )

In [ ]:
def make_hpo_trainer(max_epochs=NUM_EPOCHS):
    """Lightweight trainer for HPO — no checkpointing, no logging."""
    return L.Trainer(
        max_epochs=max_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        enable_checkpointing=False,
        logger=False,
    )

In [ ]:
COMBINATIONS = {
    "combination_1": {"d_model": 64,  "n_layers": 4, "N": 8,  "lr": 5e-4},
    "combination_2": {"d_model": 128, "n_layers": 2, "N": 16, "lr": 5e-4},
}

def run_both_combinations(dataset_name, DatasetClass, data_dict, batch_sizes):
    """
    batch_sizes: dict like {"combination_1": 8, "combination_2": 4}
    """
    train_labels = [item[1] for item in data_dict['train'].data]
    num_classes = max(train_labels) + 1

    results = {}

    for combo_name, cfg in COMBINATIONS.items():
        bs = batch_sizes[combo_name]
        print(f"\n{'='*50}\n{dataset_name} — {combo_name}: {cfg} | batch_size={bs}\n{'='*50}")

        train_dataset = DatasetClass(data_dict['train'].data)
        val_dataset   = DatasetClass(data_dict['val'].data)

        train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
        val_loader   = DataLoader(val_dataset, batch_size=bs,
                                  num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

        model = MambaClassifier(
            vocab_size=data_dict['vocab_size'],
            d_model=cfg['d_model'], n_layers=cfg['n_layers'], N=cfg['N'],
            num_classes=num_classes, lr=cfg['lr'],
        )

        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Parameters: {total_params}")

        trainer = make_trainer()
        trainer.fit(model, train_loader, val_loader)

        test_dataset = DatasetClass(data_dict['test'].data)
        test_loader  = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS,
                                  pin_memory=True, persistent_workers=True)
        trainer.test(model, test_loader)

        results[combo_name] = {
            "config": {**cfg, "batch_size": bs},
            "params": total_params,
            "val_acc": trainer.callback_metrics.get("val_acc", None),
            "test_acc": trainer.callback_metrics.get("test_acc", None),
        }

    return results

## 3.1 ListOpsDataset

In [ ]:
class ListOpsDataset(Dataset):
    def __init__(self, data, max_len=2048):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Pad or truncate
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
data_listops = torch.load('listops.pt', weights_only=False)

In [ ]:
results_listops = run_both_combinations(
    "ListOps", ListOpsDataset, data_listops,
    batch_sizes={"combination_1": 8, "combination_2": 4},
)

## 3.2 CIFAR10SequentialDataset

In [ ]:
class CIFAR10SequentialDataset(Dataset):
    """PyTorch Dataset for sequential CIFAR-10 classification."""
    def __init__(self, data, seq_len=1024):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Handle both dict and tuple formats
        if isinstance(item, dict):
            tokens = item['input_ids']
            label = item['labels']
        else:
            tokens = item[0]
            label = item[-1]

        # Convert tensors to lists if needed (for padding logic)
        if isinstance(tokens, torch.Tensor):
            tokens = tokens.tolist()
        if isinstance(label, torch.Tensor):
            label = label.item()

        # Pad or truncate to seq_len
        if len(tokens) < self.seq_len:
            mask = [1] * len(tokens) + [0] * (self.seq_len - len(tokens))
            tokens = tokens + [0] * (self.seq_len - len(tokens))
        else:
            tokens = tokens[:self.seq_len]
            mask = [1] * self.seq_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
data_cifar10_sequential = torch.load('cifar10_sequential_lra.pt', weights_only=False)

In [ ]:
results_cifar = run_both_combinations(
    "CIFAR-10", CIFAR10SequentialDataset, data_cifar10_sequential,
    batch_sizes={"combination_1": 8, "combination_2": 8},
)

# 4. Seeing results

In [ ]:
print("\n" + "=" * 70)
print("  RESULTS SUMMARY — BOTH COMBINATIONS")
print("=" * 70)

for name, results in [
    ("ListOps",   results_listops),
    ("CIFAR-10",  results_cifar),
]:
    print(f"\n  {name}")
    for combo_name, r in results.items():
        print(f"    {combo_name}: val_acc={r['val_acc']}  test_acc={r['test_acc']}  params={r['params']}  config={r['config']}")

print("\n" + "=" * 70)

In [ ]:
notebook_end_time = time.time()
total_time = notebook_end_time - notebook_start_time

# Convert to minutes and seconds for easier reading
minutes = int(total_time // 60)
seconds = int(total_time % 60)
print(f"Total 'Run All' execution time: {minutes} minutes and {seconds} seconds")